# Visualizing regression results

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
DATA_PATH = '../../../updated-data/obs/lme-ready'
NULL_DATA_PATH = '../../../updated-data/null/null-lme-ready'

PARAMS_PATH = '../../3-REGRESSION-MODELS/reports/agg-params.csv'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [3]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [4]:
def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

In [5]:
fs = [
    f for f in os.listdir(DATA_PATH)
    if (('xling-' in f) or ('callfriend' in f) or ('callhome' in f))
    and (not f.startswith('.'))
    and f.endswith('.parquet')
]
len(fs)

1289

## Importing parameters

In [6]:
dfps = pd.read_csv(PARAMS_PATH)
dfps['param'].loc[4] = 'null'
dfps.head()

,param,beta,sd,k,se,t,p
0,nx,0.151483,0.016978,3027,0.000309,490.884027,0.000000e+00
1,ny,-0.022291,0.007598,3027,0.000138,-161.414753,0.000000e+00
2,self,-0.248053,0.572784,3027,0.010411,-23.826491,0.000000e+00
3,other,-0.178894,0.224961,3027,0.004089,-43.751689,0.000000e+00
4,null,0.097085,1.020201,3027,0.018543,5.235658,8.782053e-08


## Processing individual datasets

In [7]:
COEF_VAR = 'I'

In [8]:
# param_list = ['Intercept', 'nx', 'ny', 'self']
param_list = ['nx', 'ny', 'self']

In [9]:
VAL_COL = 'resid'
group_by_cols = ['lang', 'turn_delta', 'self']

In [10]:
dataframe_cols = [
    'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
    COEF_VAR,
    'x_turn_id', 'y_turn_id', # 'AVAR',
    'nx', 'ny'
]

In [11]:
df_scale, df_regression = [], []

In [12]:
for f in tqdm(fs):

    dfo = pq.ParquetFile(os.path.join(DATA_PATH, f))
    if f.startswith('grace-'):
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f.replace('grace', 'Miao-fNIRS')))
    else:
        dfn = pq.ParquetFile(os.path.join(NULL_DATA_PATH, f))

    df = [dfo.read(columns=dataframe_cols).to_pandas()]
    df[-1]['null'] = False
    df += [dfn.read(columns=dataframe_cols).to_pandas()]
    df[-1]['null'] = True
    df = pd.concat(df, ignore_index=True)

    df = df.loc[
        (df['nx'] >= 5)
        & (df['ny'] >= 5)
    ]

    if ('CANDOR' in f.split('/')[-1]):
        df['corpus'] = 'CANDOR'

    if ('MICASE' in f.split('/')[-1]):
        df['corpus'] = 'MICASE'

    df['self'] = (df['x_speaker'] == df['y_speaker']).astype(int)
    df['self'].loc[df['null']] = 2

    # df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df['turn_delta'] = (df['y_turn_id'] - df['x_turn_id'])

    df['lang'] = lang_id(f)

    # if ('CANDOR' in f.split('/')[-1]) or ('grace' in f.split('/')[-1]):
    #     df['turn_delta'].loc[df['self'] == 0] -= 1

    if VAL_COL == 'resid':
        df['pred'] = (df['nx'] * dfps['beta'].loc[dfps['param'].isin(['nx'])].values[0]) + (df['ny'] * dfps['beta'].loc[dfps['param'].isin(['ny'])].values[0])

        df['resid'] = df[COEF_VAR] - df['pred']

    df_regression += [
        df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('mean').reset_index()
    ]

    df_regression[-1]['std'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('std').reset_index()[VAL_COL]

    df_regression[-1]['k'] = df[group_by_cols+[VAL_COL]].groupby(by=group_by_cols).agg('count').reset_index()[VAL_COL]

    df_regression[-1]['se'] = df_regression[-1]['std'] / np.sqrt(df_regression[-1]['k'])

df_regression = pd.concat(df_regression, ignore_index=True)

  0%|          | 0/1289 [00:00<?, ?it/s]

In [13]:
df0 = df_regression[['turn_delta', 'self'] + [VAL_COL]].groupby(by=['turn_delta', 'self']).agg('mean').reset_index()
df0.head()

,turn_delta,self,resid
0,1,0,-0.370194
1,1,1,-0.619612
2,1,2,-0.273884
3,2,0,-0.373491
4,2,1,-0.594051


In [14]:
df1 = df_regression[['turn_delta', 'self', 'lang'] + [VAL_COL]].groupby(by=['turn_delta', 'self', 'lang']).agg('mean').reset_index()
df1.head(n=25)

,turn_delta,self,lang,resid
0,1,0,cro,-0.320393
1,1,0,deu,-0.296094
2,1,0,eng,-0.239327
3,1,0,fin,-0.208659
4,1,0,fra,-0.298343
5,1,0,ko,-0.355124
6,1,0,spa,-0.321093
7,1,0,yid,-2.383450
8,1,0,zho,-0.872273
9,1,1,cro,-0.361610


## Plotly vis

In [15]:
import plotly.graph_objects as go

### Aggregate/total visualization

In [16]:
sel_1 = (df0['turn_delta'] <= 200) & (df0['turn_delta'] > 0) & ((df0['turn_delta'] % 2) != 0)

In [17]:
fig = go.Figure()

In [18]:
sel = df0['self'] == 1
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='self',
        showlegend=True,
        legendgroup='self',
        line = dict(
            color='orange'
        )
    )
)

sel = df0['self'] == 0
fig.add_trace(
    go.Scatter(
        y = df0[VAL_COL].loc[sel & sel_1].values,
        # x = np.array(range(int((sel & sel_1).sum()))) + 1,
        x = df0['turn_delta'].loc[sel & sel_1].values,
        mode='lines',
        name='other',
        showlegend=True,
        legendgroup='other',
        line = dict(
            color='royalblue'
        )
    )
)

fig.update_layout(
    template='xgridoff',
    yaxis_title='Residual I(x;y)',
    xaxis_title='Distance in turns from x to y'
)

In [19]:
fig.write_html(os.path.join(OUTPUTS_PATH, 'all-corpora.html'))
fig.write_image(os.path.join(OUTPUTS_PATH, 'all-corpora.png'), scale=5)

### Per corpus vis

In [23]:
figs = []

In [24]:
for corpus in df1.lang.unique():
    sub = df1.loc[df1['lang'].isin([corpus])]

    fig = go.Figure()

    sel_t = (sub['turn_delta'] > 0) & (sub['turn_delta'] <= 200)

    sel = sub['self'] == 1
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='self',
            showlegend=True,
            legendgroup='self',
            line = dict(
                color='orange'
            )
        )
    )

    sel = sub['self'] == 0
    fig.add_trace(
        go.Scatter(
            x=sub['turn_delta'].loc[sel & sel_t].values,
            y = sub[VAL_COL].loc[sel & sel_t].values,
            mode='lines',
            name='other',
            showlegend=True,
            legendgroup='other',
            line = dict(
                color='royalblue'
            )
        )
    )

    fig.update_layout(template='xgridoff')

    figs += [fig]

In [25]:
for corpus, plot in zip(df1.lang.unique(), figs):
    print(corpus)
    plot.show()
    plot.write_html(os.path.join(OUTPUTS_PATH, corpus+'.html'))
    plot.write_image(os.path.join(OUTPUTS_PATH, corpus+'.png'), scale=5)

cro


deu


eng


fin


fra


ko


spa


yid


zho
